# 06 ml dataset preparation

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
# Combine independently generated features with ground truth

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = PUBLICATION_ROOT
OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"

# 1. Load ground-truth dataset
ground_truth = pd.read_csv(
    OUTPUT_DIR / "session_ground_truth.csv"
)

# Preserve binary labels as integers while allowing missing values
ground_truth["GroundTruthLabel"] = pd.to_numeric(
    ground_truth["GroundTruthLabel"],
    errors="coerce"
).astype("Int64")

ground_truth["IntendedLabel"] = pd.to_numeric(
    ground_truth["IntendedLabel"],
    errors="coerce"
).astype("Int64")


# 2. Load independently generated behavioural features
features = pd.read_csv(
    OUTPUT_DIR / "session_features_jupyter.csv"
)


# 3. Merge features with ground truth using SessionID
ml_data = ground_truth.merge(
    features,
    on="SessionID",
    how="left",
    validate="one_to_one"
)


# 4. Validate the merged dataset
assert len(ml_data) == 1200
assert ml_data["SessionID"].is_unique
assert ml_data["GroundTruthLabel"].dropna().isin([0, 1]).all()

print("Total sessions:", len(ml_data))
print("Total columns:", len(ml_data.columns))


# 5. Identify ground-truth labels and eligibility fields
relevant_columns = [
    col for col in ground_truth.columns
    if any(
        term in col.lower()
        for term in [
            "label",
            "malicious",
            "scenario",
            "eligible",
            "cohort",
            "partition"
        ]
    )
]

print("\nGround-truth label and eligibility columns:")
print(relevant_columns)


# 6. Display sample records
display(
    ml_data[
        ["SessionID"] + relevant_columns
    ].head()
)

In [ ]:
# Examine ground-truth label distribution and missing labels

print("Ground-truth columns:")
print(ground_truth.columns.tolist())

print("\nOverall ground-truth label distribution:")
print(ml_data["GroundTruthLabel"].value_counts(dropna=False))

# Summarise labels by scenario
label_summary = (
    ml_data.groupby("ScenarioID")
    .agg(
        TotalSessions=("SessionID", "size"),
        VerifiedLabels=("GroundTruthLabel", lambda x: x.notna().sum()),
        MissingLabels=("GroundTruthLabel", lambda x: x.isna().sum()),
        BenignLabels=("GroundTruthLabel", lambda x: (x == 0).sum()),
        MaliciousLabels=("GroundTruthLabel", lambda x: (x == 1).sum())
    )
    .reset_index()
)

print("\nGround-truth labels by scenario:")
display(label_summary)

print("\nTotal sessions:", len(ml_data))
print("Sessions with labels:", ml_data["GroundTruthLabel"].notna().sum())
print("Sessions without labels:", ml_data["GroundTruthLabel"].isna().sum())

In [ ]:
# Examine session eligibility and experiment partitions

reference_file = (
    PROJECT_ROOT / "data" / "processed" / "session_features.csv"
)

eligibility_columns = [
    "SessionID",
    "FeatureMLTrainingEligible",
    "FeatureOriginal14DayCohortEligible",
    "FeatureExtendedCohortEligible",
    "ExperimentPartition",
    "ExperimentIncluded",
    "FeatureReviewReasons"
]

# Read eligibility metadata from the reference implementation
eligibility = pd.read_csv(
    reference_file,
    usecols=eligibility_columns,
    dtype=str
)

# Attach eligibility information to the labelled sessions
eligibility_audit = ml_data[
    ["SessionID", "GroundTruthLabel"]
].merge(
    eligibility,
    on="SessionID",
    how="left",
    validate="one_to_one"
)

print("ML training eligibility:")
print(
    eligibility_audit[
        "FeatureMLTrainingEligible"
    ].value_counts(dropna=False)
)

print("\nEligibility by ground-truth label:")

display(
    pd.crosstab(
        eligibility_audit["FeatureMLTrainingEligible"],
        eligibility_audit["GroundTruthLabel"].astype("string").fillna("Missing"),
        margins=True
    )
)

print("\nExperiment partitions:")
print(
    eligibility_audit[
        "ExperimentPartition"
    ].value_counts(dropna=False)
)

print("\nExperiment inclusion:")
print(
    eligibility_audit[
        "ExperimentIncluded"
    ].value_counts(dropna=False)
)

In [ ]:
# Select labelled sessions eligible for the ML experiment

# Merge the independently generated dataset with
# Codex's existing eligibility and partition metadata
experiment_data = ml_data.merge(
    eligibility[
        [
            "SessionID",
            "FeatureMLTrainingEligible",
            "ExperimentIncluded",
            "ExperimentPartition"
        ]
    ],
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Examine the relationship between eligibility and inclusion
print("Eligibility versus experiment inclusion:")
print(
    pd.crosstab(
        experiment_data["FeatureMLTrainingEligible"],
        experiment_data["ExperimentIncluded"]
    )
)

# Select sessions with verified labels that satisfy both flags
valid_sessions = (
    experiment_data["GroundTruthLabel"].notna()
    & experiment_data["FeatureMLTrainingEligible"].eq("true")
    & experiment_data["ExperimentIncluded"].eq("true")
)

ml_cohort = experiment_data.loc[valid_sessions].copy()

print("\nTotal sessions:", len(experiment_data))
print("Selected experiment sessions:", len(ml_cohort))
print("Excluded sessions:", len(experiment_data) - len(ml_cohort))

# Examine the actual partition sizes after exclusions
print("\nSelected sessions by partition:")
print(
    ml_cohort["ExperimentPartition"].value_counts()
)

# Check benign and malicious labels in each partition
print("\nClass distribution by partition:")
display(
    pd.crosstab(
        ml_cohort["ExperimentPartition"],
        ml_cohort["GroundTruthLabel"],
        margins=True
    )
)

In [ ]:
# Examine feature availability and variation in training data

feature_columns = [
    col for col in features.columns
    if col != "SessionID"
]

# Use training data only to make feature selection decisions
train_data = ml_cohort.loc[
    ml_cohort["ExperimentPartition"] == "train",
    feature_columns
].apply(pd.to_numeric, errors="coerce")

feature_audit = pd.DataFrame({
    "MissingCount": train_data.isna().sum(),
    "MissingPercent": (
        train_data.isna().mean() * 100
    ).round(2),
    "UniqueValues": train_data.nunique(dropna=True),
    "NonZeroCount": (
        train_data.fillna(0) != 0
    ).sum()
})

# Identify features requiring exclusion or review
def assess_feature(row):
    if row.name == "protected_resource_event_count":
        return "Exclude - diagnostic"

    if row["MissingPercent"] == 100:
        return "Exclude - all missing"

    if row["UniqueValues"] <= 1:
        return "Review - no variation"

    return "Candidate"

feature_audit["Assessment"] = feature_audit.apply(
    assess_feature,
    axis=1
)

feature_audit = feature_audit.sort_values(
    ["MissingPercent", "UniqueValues"],
    ascending=[False, True]
)

print("Total behavioural features:", len(feature_columns))
print("Training sessions:", len(train_data))

print("\nFeature assessment:")
print(feature_audit["Assessment"].value_counts())

display(feature_audit)

In [ ]:
# Prepare feature matrices and labels for machine learning

# Select features using training data assessment only
selected_features = feature_audit.index[
    feature_audit["Assessment"] == "Candidate"
].tolist()

print("Selected behavioural features:", len(selected_features))
print("\nSelected features:")
for feature in selected_features:
    print("-", feature)


# Separate the chronological experiment partitions
train = ml_cohort[
    ml_cohort["ExperimentPartition"] == "train"
].copy()

validation = ml_cohort[
    ml_cohort["ExperimentPartition"] == "validation"
].copy()

test = ml_cohort[
    ml_cohort["ExperimentPartition"] == "test"
].copy()


# Prepare numerical feature matrices
X_train = train[selected_features].apply(
    pd.to_numeric, errors="coerce"
)

X_val = validation[selected_features].apply(
    pd.to_numeric, errors="coerce"
)

X_test = test[selected_features].apply(
    pd.to_numeric, errors="coerce"
)


# Prepare binary ground-truth labels
y_train = train["GroundTruthLabel"].astype(int)
y_val = validation["GroundTruthLabel"].astype(int)
y_test = test["GroundTruthLabel"].astype(int)


# Display dataset dimensions
print("\nTraining:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nMissing feature values:")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Testing:", X_test.isna().sum().sum())

In [ ]:
# Preprocess the ML datasets without external scaling

from sklearn.impute import SimpleImputer
import pandas as pd

# Fit the imputer on training data only
imputer = SimpleImputer(strategy="median")

X_train_ready = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Apply training medians to validation and testing
X_val_ready = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_ready = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Training:", X_train_ready.shape)
print("Validation:", X_val_ready.shape)
print("Testing:", X_test_ready.shape)

print("\nRemaining missing values:")
print("Training:", X_train_ready.isna().sum().sum())
print("Validation:", X_val_ready.isna().sum().sum())
print("Testing:", X_test_ready.isna().sum().sum())

In [ ]:
# Save the prepared ML datasets for model training

ml_output_dir = OUTPUT_DIR / "ml_ready"
ml_output_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "train": (train, X_train_ready, y_train),
    "validation": (validation, X_val_ready, y_val),
    "test": (test, X_test_ready, y_test)
}

for name, (sessions, X, y) in datasets.items():

    dataset = pd.concat(
        [
            sessions[["SessionID"]],
            X,
            y.rename("GroundTruthLabel")
        ],
        axis=1
    )

    dataset.to_csv(
        ml_output_dir / f"{name}_ml.csv",
        index=False
    )

    print(f"{name}_ml.csv: {dataset.shape}")

print("\nDatasets saved to:", ml_output_dir)